# DeepSeek-MoE-16B Routing Trace Extraction (v2 — full rich parity with OLMoE)

Generates `deepseek_routing_trace.json` (same 12 domain-balanced prompts as
`extract_routing_trace.ipynb`, for direct model-to-model comparison) plus
`deepseek_routing_trace_umap.json`.

**v2 upgrade — full parity.** This notebook now emits the **same rich deep-extraction schema as
`extract_routing_trace.ipynb`** (OLMoE): per-layer `layer_flow` (Q/K/V/O weight downsamples,
per-head Q/K/V with RoPE, attention maps, residuals, RMSNorm), `router_matrices`,
`hidden_vectors`, `expert_weights`, `expert_outputs`, `embed_strip` — so the frontend's deep
"click a block → full matrix arithmetic" modals work identically for DeepSeek. FFN routing is at
the top level as `layers` (OLMoE-style), each MoE-layer entry carrying `top_experts`,
`top_weights` (RAW full-softmax top-k, **not** renormalized — same convention as OLMoE), and the
full 64-expert `all_probs`.

**DeepSeek's two divergences** (see `docs/model-architecture-jetmoe-deepseek-research.md` §3),
now fully captured:
1. **2 shared experts fire unconditionally on every token**, summed additively (gate-free) with
   the routed output — extracted as `shared_expert_weights` + per-token `shared_expert_outputs`.
2. **Layer 0 is a plain dense FFN** (intermediate 10944, no router) — its routing `layers[0]` is
   `is_dense: true`/`tokens: null`, and its FFN weights + per-token outputs are in `dense_ffn`.

Attention is standard MHA (16 heads × 128, no MoA) so `layer_flow` mirrors OLMoE minus the
q/k-norm step (`has_qk_norm: false`). A `reconcile=True` smoke pass self-validates the untestable
deep recompute (only the correct combine convention reproduces the real MoE delta).

DeepSeek-MoE-16B requires `trust_remote_code=True`. Run on a Colab A100 **40GB** GPU runtime
(16.4B params, ~33GB in bf16). Uses `attn_implementation="eager"` so `output_attentions` returns
the per-head attention maps the deep modal needs.


In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


if importlib.util.find_spec("torch") is None:
    pip_install("torch")

# transformers PINNED TO THE EXACT VERSION -- not a range -- and this isn't a style choice,
# it's the second compatibility break found the hard way. DeepSeek's trust_remote_code
# modeling_deepseek.py is 2024-era code with two confirmed hard dependencies on APIs that
# later transformers releases removed: (1) `from transformers.utils.import_utils import
# is_torch_fx_available`, present through 4.57.1, gone at the 5.0.0 major bump; (2)
# `DynamicCache.get_usable_length(...)`, called during attention's cache handling on every
# forward pass (not just multi-step generation) -- present through 4.53.0, gone by 4.55.0,
# entirely within the 4.x line. A range like ">=4.36.2,<5.0.0" lets pip resolve to the
# newest 4.x (4.57.1) and hits break #2 immediately. Rather than keep narrowing the upper
# bound against whatever the next undiscovered break turns out to be, pin the EXACT version
# DeepSeek's own requirements.txt states as its floor (4.36.2) -- almost certainly what the
# authors actually tested against, and confirmed here to have both APIs intact.
pip_install("transformers==4.36.2", "accelerate", "umap-learn", "numpy", "scikit-learn")

print("Dependency installation complete.")

In [ ]:
import json
import os
import sys
import types
import importlib.machinery
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

# transformers 4.36.2's trust_remote_code loader (check_imports in dynamic_module_utils.py)
# does a naive static regex scan of EVERY top-level import statement in modeling_deepseek.py
# and demands each one be importable -- it only skips imports inside a try/except block, and
# flash_attn is imported under `if is_flash_attn_2_available():` (an if-guard, not
# try/except), so the scanner isn't smart enough to skip it even though it's never actually
# needed for a single eager/sdpa forward pass. Stub it in sys.modules to satisfy that static
# scan (empirically verified against the real check_imports/get_imports regex logic). The
# stub needs a real __spec__ -- a bare types.ModuleType() leaves __spec__ as None, and
# importlib.util.find_spec() raises ValueError (not a clean False) when a sys.modules entry
# has __spec__ = None, which would crash transformers' own is_flash_attn_2_available() the
# moment it runs. With a real (but loader=None) ModuleSpec, find_spec() returns cleanly, and
# is_flash_attn_2_available() still correctly resolves to False afterward via
# importlib.metadata.version("flash_attn") raising PackageNotFoundError (no real distribution
# installed) -- so the `if` guard still evaluates False and eager/sdpa attention runs as
# normal; this stub cannot cause a silent switch to a broken flash-attn code path.
if "flash_attn" not in sys.modules:
    stub = types.ModuleType("flash_attn")
    stub.__spec__ = importlib.machinery.ModuleSpec("flash_attn", loader=None)
    sys.modules["flash_attn"] = stub

MODEL_ID = "deepseek-ai/deepseek-moe-16b-base"
OUT_PATH = "deepseek_routing_trace.json"
UMAP_OUT_PATH = "deepseek_routing_trace_umap.json"
TOP_K_NEXT_TOKEN = 50

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# DeepSeek's config.json and modeling_deepseek.py are co-authored in the same HF repo (no
# mainline-transformers version-drift risk like JetMoE's config.json has), but verify
# against the paper/config numbers anyway before spending GPU time loading a 16B model on a
# wrong assumption.
EXPECTED = {
    "num_hidden_layers": 28,
    "n_routed_experts": 64,
    "num_experts_per_tok": 6,
    "n_shared_experts": 2,
    "first_k_dense_replace": 1,
    "moe_layer_freq": 1,
    "hidden_size": 2048,
    "norm_topk_prob": False,
}

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
mismatches = {k: (getattr(config, k, "<missing>"), v) for k, v in EXPECTED.items() if getattr(config, k, None) != v}
assert not mismatches, (
    f"DeepSeek config mismatch vs. docs/model-architecture-jetmoe-deepseek-research.md: {mismatches} "
    f"-- investigate before proceeding (this repo's config.json and modeling code are "
    f"co-authored, so a mismatch here would be unexpected rather than the routine drift risk "
    f"JetMoE has)."
)
print("DeepSeek config verified:", {k: getattr(config, k) for k in EXPECTED})

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,  # not `dtype=` -- that alias postdates the <5.0.0 pin above; the
    # model card's own example uses torch_dtype, confirmed present in 4.36.2's from_pretrained.
    attn_implementation="eager",  # eager so output_attentions returns per-head attention maps
    device_map="auto",
    output_loading_info=True,
)
model.eval()

assert not loading_info["missing_keys"], (
    f"Some model weights were NOT loaded from the checkpoint (randomly initialized "
    f"instead): {loading_info['missing_keys']}"
)
print(f"unexpected_keys (informational): {loading_info.get('unexpected_keys', [])}")

num_layers = config.num_hidden_layers
hidden_size = config.hidden_size
num_experts = config.n_routed_experts
top_k_experts = config.num_experts_per_tok
n_shared_experts = config.n_shared_experts

# DeepseekDecoderLayer.__init__ picks DeepseekMoE (has .gate) vs. plain DeepseekMLP (dense,
# no .gate) per-layer based on first_k_dense_replace/moe_layer_freq -- read it off the
# actually-instantiated modules rather than re-deriving the condition, so this stays correct
# even if the config values above turn out to be wrong in some future checkpoint.
moe_layer_indices = [li for li in range(num_layers) if hasattr(model.model.layers[li].mlp, "gate")]
dense_layer_indices = [li for li in range(num_layers) if li not in moe_layer_indices]
print(f"Loaded {MODEL_ID}: {num_layers} layers, {len(dense_layer_indices)} dense {dense_layer_indices}, "
      f"{len(moe_layer_indices)} MoE (top-{top_k_experts} of {num_experts}, {n_shared_experts} always-on shared)")
assert dense_layer_indices == [0], f"Expected only layer 0 to be dense, got {dense_layer_indices}"

In [ ]:
# ---- extraction helpers (downsampling), mirrored verbatim from extract_routing_trace.ipynb so
# the frontend math modals render DeepSeek grids at the same resolution as OLMoE ----
def to_float(t):
    return t.detach().float().cpu()


def downsample_1d(vec, buckets):
    n = vec.shape[0]
    idx = torch.linspace(0, n, buckets + 1).round().long()
    return [round(vec[idx[i]: max(idx[i] + 1, idx[i + 1])].mean().item(), 5) for i in range(buckets)]


def downsample_2d(mat, rows, cols):
    R, C = mat.shape
    ridx = torch.linspace(0, R, rows + 1).round().long()
    cidx = torch.linspace(0, C, cols + 1).round().long()
    out = []
    for i in range(rows):
        r0, r1 = ridx[i].item(), max(ridx[i].item() + 1, ridx[i + 1].item())
        row_vals = []
        for j in range(cols):
            c0, c1 = cidx[j].item(), max(cidx[j].item() + 1, cidx[j + 1].item())
            row_vals.append(round(mat[r0:r1, c0:c1].mean().item(), 5))
        out.append(row_vals)
    return out


# downsample resolutions -- identical to extract_routing_trace.ipynb
ROUTER_GRID = (10, 12)   # W_router downsample for the router diagram
HIDDEN_STRIP = 20        # hidden-vector strip length
EXPERT_GRID = (5, 5)     # per-expert gate/up/down weight downsample
ATTN_GRID = (10, 10)     # per Q/K/V/O weight-matrix downsample
HEAD_STRIP = 6           # single-head Q/K/V slice downsample width

# attention dims + a hard MHA check: the per-head K/V slicing below (hh*head_dim) assumes
# num_key_value_heads == num_attention_heads (standard MHA). deepseek-moe-16b-base is 16/16 per
# the research doc; assert rather than silently mis-slice a GQA checkpoint.
num_heads_attn = config.num_attention_heads
num_kv_heads = getattr(config, "num_key_value_heads", num_heads_attn)
head_dim = getattr(config, "head_dim", None) or (hidden_size // num_heads_attn)
moe_intermediate_size = config.moe_intermediate_size
rope_theta = getattr(config, "rope_theta", 10000.0)
assert num_kv_heads == num_heads_attn, (
    f"Extraction assumes MHA (kv_heads==attn_heads); got {num_kv_heads} vs {num_heads_attn} -- "
    f"add repeat_kv-style head mapping before running on a GQA checkpoint."
)
print(f"attention: {num_heads_attn} heads x {head_dim} dim (MHA); "
      f"moe_intermediate_size={moe_intermediate_size}; rope_theta={rope_theta}")

# Fail fast if the non-obvious module paths this notebook depends on (DeepseekMoE.shared_experts,
# experts[i].{gate,up,down}_proj, self_attn.{q,k,v,o}_proj) differ from the modeling source --
# catch it on layer 0 / the first MoE layer and print the real attribute names.
_mi = moe_layer_indices[0]
_moe = model.model.layers[_mi].mlp
assert hasattr(_moe, "shared_experts") and hasattr(_moe, "experts") and hasattr(_moe, "gate"), \
    f"expected DeepseekMoE.gate/experts/shared_experts; mlp attrs={list(dict(_moe.named_children()))}"
for _name in ("gate_proj", "up_proj", "down_proj"):
    assert hasattr(_moe.experts[0], _name), f"expected experts[0].{_name}; attrs={list(dict(_moe.experts[0].named_children()))}"
    assert hasattr(_moe.shared_experts, _name), f"expected shared_experts.{_name}; attrs={list(dict(_moe.shared_experts.named_children()))}"
_attn0 = model.model.layers[0].self_attn
for _name in ("q_proj", "k_proj", "v_proj", "o_proj"):
    assert hasattr(_attn0, _name), f"expected self_attn.{_name}; attrs={list(dict(_attn0.named_children()))}"
_dense0 = model.model.layers[dense_layer_indices[0]].mlp
assert hasattr(_dense0, "gate_proj") and _dense0.gate_proj.weight.shape[0] == 10944, \
    f"expected dense layer DeepseekMLP gate_proj @10944; got shape {tuple(_dense0.gate_proj.weight.shape)}"
print("verified module paths: mlp.{gate,experts[i].gate/up/down_proj,shared_experts}, self_attn.{q,k,v,o}_proj, "
      f"dense_ffn intermediate={_dense0.gate_proj.weight.shape[0]}")


In [ ]:
# 6 domains x 2 short, trivia/completion-style prompts each = 12 total -- identical set to
# extract_routing_trace.ipynb's PROMPTS, so results are directly comparable across models.
PROMPTS = {
    "code": [
        "The most popular programming language for data science is",
        "A loop that never terminates is called an infinite",
    ],
    "math": [
        "The square root of sixteen is",
        "Two plus two equals",
    ],
    "biomedical": [
        "The organ that pumps blood throughout the human body is the",
        "White blood cells are a key part of the body's immune",
    ],
    "legal": [
        "The document that establishes the fundamental laws of the United States is called the",
        "A person accused of a crime is presumed",
    ],
    "creative_writing": [
        "It was a dark and stormy",
        "Roses are red, violets are",
    ],
    "conversational": [
        "Thank you so much, I really appreciate",
        "It was great catching up, see you",
    ],
}

domains = list(PROMPTS.keys())
total_prompts = sum(len(v) for v in PROMPTS.values())
print(f"Domains: {domains}")
print(f"Total prompts: {total_prompts}")
assert total_prompts == 12, f"Expected 12 prompts (6 x 2), got {total_prompts}"
for domain, examples in PROMPTS.items():
    assert len(examples) == 2, f"{domain} has {len(examples)} prompts, expected 2"

In [ ]:
def extract_for_prompt(prompt, reconcile=False):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    token_ids = inputs["input_ids"][0].tolist()
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    seq_len = len(token_ids)

    # hooks: router input (MoE layers), the two RMSNorms, Q/K/V projections, and the whole
    # attention module's output -- mirrors extract_routing_trace.ipynb's hook set, adapted to
    # DeepSeek module names (self_attn, no q_norm/k_norm).
    gate_inputs, ln1_outputs, ln2_outputs = {}, {}, {}
    q_raw, k_raw, v_raw, attn_outputs = {}, {}, {}, {}
    hooks = []

    def make_pre_hook(store, li):
        def hook(module, args):
            store[li] = to_float(args[0])  # [1, seq, *]
        return hook

    def make_post_hook(store, li):
        def hook(module, args, output):
            val = output[0] if isinstance(output, tuple) else output
            store[li] = val[0].detach().float().cpu()  # drop batch dim -> [seq, *]
        return hook

    for li in range(num_layers):
        layer = model.model.layers[li]
        hooks.append(layer.input_layernorm.register_forward_hook(make_post_hook(ln1_outputs, li)))
        hooks.append(layer.post_attention_layernorm.register_forward_hook(make_post_hook(ln2_outputs, li)))
        hooks.append(layer.self_attn.q_proj.register_forward_hook(make_post_hook(q_raw, li)))
        hooks.append(layer.self_attn.k_proj.register_forward_hook(make_post_hook(k_raw, li)))
        hooks.append(layer.self_attn.v_proj.register_forward_hook(make_post_hook(v_raw, li)))
        hooks.append(layer.self_attn.register_forward_hook(make_post_hook(attn_outputs, li)))
        if li in moe_layer_indices:
            hooks.append(layer.mlp.gate.register_forward_pre_hook(make_pre_hook(gate_inputs, li)))

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True, output_hidden_states=True)

    for h in hooks:
        h.remove()

    assert set(gate_inputs) == set(moe_layer_indices), (
        f"Gate hooks fired for layers {sorted(gate_inputs)}, expected {moe_layer_indices}"
    )
    # The per-head attention maps come from outputs.attentions, which only works if this
    # trust_remote_code build honours attn_implementation="eager" + output_attentions=True.
    # Fail fast with a clear cause rather than a cryptic NoneType-subscript deep in the layer loop.
    assert outputs.attentions is not None and len(outputs.attentions) == num_layers and \
        outputs.attentions[0].shape[1] == num_heads_attn, (
        f"output_attentions did not return per-head maps (got "
        f"{None if outputs.attentions is None else tuple(outputs.attentions[0].shape)}). This "
        f"trust_remote_code 4.36.2 build may ignore attn_implementation='eager' -- fallback: "
        f"recompute maps from the per-head q/k (as the JetMoE notebook does) and drop output_attentions."
    )

    # ---- FFN routing per layer (dense layer 0 -> tokens None). Recompute the full softmax from
    # the hooked gate input + gate weight (MoEGate.forward only returns topk, not all 64 probs).
    # norm_topk_prob=False, so top_weights are raw softmax scores -- same convention as OLMoE. ----
    silu = torch.nn.functional.silu
    ffn_layers_trace = []
    for li in range(num_layers):
        if li not in moe_layer_indices:
            ffn_layers_trace.append({"layer": li, "is_dense": True, "shared_experts_active": False, "tokens": None})
            continue
        h = gate_inputs[li][0]  # [seq, hidden]
        gate_weight = to_float(model.model.layers[li].mlp.gate.weight)  # [num_experts, hidden]
        logits = torch.nn.functional.linear(h, gate_weight, None)
        assert logits.shape == (seq_len, num_experts), (
            f"layer {li}: logits shape {tuple(logits.shape)}, expected ({seq_len}, {num_experts})"
        )
        probs = torch.softmax(logits, dim=-1)
        topk = torch.topk(probs, k=top_k_experts, dim=-1)
        tokens_trace = []
        for t in range(seq_len):
            tokens_trace.append({
                "token_index": t,
                "top_experts": topk.indices[t].tolist(),
                "top_weights": [round(w, 5) for w in topk.values[t].tolist()],
                "all_probs": [round(p, 5) for p in probs[t].tolist()],
            })
        ffn_layers_trace.append({"layer": li, "is_dense": False, "shared_experts_active": True, "tokens": tokens_trace})

    # ---- next-token candidates ----
    next_token_logits = outputs.logits[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits.float(), dim=-1)
    top_next = torch.topk(next_token_probs, k=TOP_K_NEXT_TOKEN)
    next_token_candidates = [
        {"token": tokenizer.decode([tid]), "prob": round(p, 6)}
        for p, tid in zip(top_next.values.tolist(), top_next.indices.tolist())
    ]

    # ---- router weight downsamples + hidden vectors (per layer; dense layer -> null router) ----
    router_matrices, hidden_vectors = [], []
    for li in range(num_layers):
        if li in moe_layer_indices:
            router_matrices.append(downsample_2d(to_float(model.model.layers[li].mlp.gate.weight), *ROUTER_GRID))
            h = gate_inputs[li][0]
        else:
            router_matrices.append(None)
            h = ln2_outputs[li]  # dense FFN input = post-attention RMSNorm output
        hidden_vectors.append([downsample_1d(h[t], HIDDEN_STRIP) for t in range(seq_len)])

    # ---- routed-expert weights + recomputed outputs (only experts actually selected somewhere) ----
    active_pairs = sorted(
        {(lt["layer"], e) for lt in ffn_layers_trace if lt["tokens"] for tt in lt["tokens"] for e in tt["top_experts"]}
    )
    expert_weights, weight_cache = {}, {}
    for li, e in active_pairs:
        exp = model.model.layers[li].mlp.experts[e]
        gate_w, up_w, down_w = to_float(exp.gate_proj.weight), to_float(exp.up_proj.weight), to_float(exp.down_proj.weight)
        weight_cache[(li, e)] = (gate_w, up_w, down_w)
        expert_weights[f"{li}_{e}"] = {
            "gate": downsample_2d(gate_w, *EXPERT_GRID),
            "up": downsample_2d(up_w, *EXPERT_GRID),
            "down": downsample_2d(down_w, *EXPERT_GRID),
        }
    expert_outputs = {}
    for lt in ffn_layers_trace:
        if not lt["tokens"]:
            continue
        li = lt["layer"]
        h_layer = gate_inputs[li][0]
        for t, tt in enumerate(lt["tokens"]):
            h_t = h_layer[t]
            for e in tt["top_experts"]:
                gate_w, up_w, down_w = weight_cache[(li, e)]
                out = torch.nn.functional.linear(silu(torch.nn.functional.linear(h_t, gate_w)) * torch.nn.functional.linear(h_t, up_w), down_w)
                expert_outputs[f"{t}_{li}_{e}"] = downsample_1d(out, HIDDEN_STRIP)

    # ---- shared experts (2, always-on): one DeepseekMLP per MoE layer; recompute per-token
    # output for the always-on lane + the gate-free additive term in the combined-output modal ----
    def mlp_forward(m, h_t):
        return torch.nn.functional.linear(silu(torch.nn.functional.linear(h_t, to_float(m.gate_proj.weight))) * torch.nn.functional.linear(h_t, to_float(m.up_proj.weight)), to_float(m.down_proj.weight))

    shared_expert_weights, shared_expert_outputs = {}, {}
    for li in moe_layer_indices:
        sh = model.model.layers[li].mlp.shared_experts
        shared_expert_weights[str(li)] = {
            "gate": downsample_2d(to_float(sh.gate_proj.weight), *EXPERT_GRID),
            "up": downsample_2d(to_float(sh.up_proj.weight), *EXPERT_GRID),
            "down": downsample_2d(to_float(sh.down_proj.weight), *EXPERT_GRID),
            "intermediate_size": sh.gate_proj.weight.shape[0],
        }
        h_layer = gate_inputs[li][0]
        for t in range(seq_len):
            shared_expert_outputs[f"{t}_{li}"] = downsample_1d(mlp_forward(sh, h_layer[t]), HIDDEN_STRIP)

    # ---- dense layer(s) (layer 0): a single DeepseekMLP FFN (intermediate 10944), no router ----
    dense_ffn = {}
    for li in dense_layer_indices:
        m = model.model.layers[li].mlp
        outs = {str(t): downsample_1d(mlp_forward(m, ln2_outputs[li][t]), HIDDEN_STRIP) for t in range(seq_len)}
        dense_ffn[str(li)] = {
            "gate": downsample_2d(to_float(m.gate_proj.weight), *EXPERT_GRID),
            "up": downsample_2d(to_float(m.up_proj.weight), *EXPERT_GRID),
            "down": downsample_2d(to_float(m.down_proj.weight), *EXPERT_GRID),
            "intermediate_size": m.gate_proj.weight.shape[0],
            "outputs": outs,
        }

    # ---- embeddings ----
    hidden_states_all = [h[0].detach().float().cpu() for h in outputs.hidden_states]
    embed_vec = to_float(model.model.embed_tokens.weight)[torch.tensor(token_ids)]
    embed_strip = [downsample_1d(embed_vec[t], HIDDEN_STRIP) for t in range(seq_len)]

    # ---- RoPE (DeepSeek: standard rotate_half, base = rope_theta) ----
    def rope_cos_sin(sl, hd, base):
        inv_freq = 1.0 / (base ** (torch.arange(0, hd, 2, dtype=torch.float32) / hd))
        freqs = torch.outer(torch.arange(sl, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()

    def rotate_half(x):
        x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)

    def apply_rope(x, cos, sin):
        return x * cos + rotate_half(x) * sin

    rope_cos, rope_sin = rope_cos_sin(seq_len, head_dim, rope_theta)

    # ---- per-layer attention + residual flow (all layers; standard MHA, no q/k norm) ----
    per_layer_flow = []
    for li in range(num_layers):
        layer = model.model.layers[li]
        attn = layer.self_attn

        layer_in = hidden_states_all[li]
        layer_out = hidden_states_all[li + 1]
        attn_out = attn_outputs[li]
        after_attn_residual = layer_in + attn_out
        moe_out = layer_out - after_attn_residual

        attn_probs_all_heads = [
            [[round(v, 5) for v in row] for row in outputs.attentions[li][0, hh].detach().float().cpu().tolist()]
            for hh in range(num_heads_attn)
        ]

        q_by_head, k_by_head, v_by_head, head_output_by_head = [], [], [], []
        q_by_head_prerope, k_by_head_prerope = [], []
        for hh in range(num_heads_attn):
            q_head = q_raw[li][:, hh * head_dim:(hh + 1) * head_dim]
            k_head = k_raw[li][:, hh * head_dim:(hh + 1) * head_dim]  # MHA: kv heads == q heads
            v_head = v_raw[li][:, hh * head_dim:(hh + 1) * head_dim]
            q_rope = apply_rope(q_head, rope_cos, rope_sin)
            k_rope = apply_rope(k_head, rope_cos, rope_sin)
            q_by_head_prerope.append([downsample_1d(q_head[t], HEAD_STRIP) for t in range(seq_len)])
            k_by_head_prerope.append([downsample_1d(k_head[t], HEAD_STRIP) for t in range(seq_len)])
            q_by_head.append([downsample_1d(q_rope[t], HEAD_STRIP) for t in range(seq_len)])
            k_by_head.append([downsample_1d(k_rope[t], HEAD_STRIP) for t in range(seq_len)])
            v_by_head.append([downsample_1d(v_head[t], HEAD_STRIP) for t in range(seq_len)])
            head_out = outputs.attentions[li][0, hh].detach().float().cpu() @ v_head
            head_output_by_head.append([downsample_1d(head_out[t], HEAD_STRIP) for t in range(seq_len)])

        per_layer_flow.append({
            "ln1_weight": downsample_1d(to_float(layer.input_layernorm.weight), HIDDEN_STRIP),
            "ln1_out": [downsample_1d(ln1_outputs[li][t], HIDDEN_STRIP) for t in range(seq_len)],
            "q_weight": downsample_2d(to_float(attn.q_proj.weight), *ATTN_GRID),
            "k_weight": downsample_2d(to_float(attn.k_proj.weight), *ATTN_GRID),
            "v_weight": downsample_2d(to_float(attn.v_proj.weight), *ATTN_GRID),
            "o_weight": downsample_2d(to_float(attn.o_proj.weight), *ATTN_GRID),
            "q_raw": [downsample_1d(q_raw[li][t], HIDDEN_STRIP) for t in range(seq_len)],
            "k_raw": [downsample_1d(k_raw[li][t], HIDDEN_STRIP) for t in range(seq_len)],
            "v_raw": [downsample_1d(v_raw[li][t], HIDDEN_STRIP) for t in range(seq_len)],
            "attn_output": [downsample_1d(attn_out[t], HIDDEN_STRIP) for t in range(seq_len)],
            "attn_probs_all_heads": attn_probs_all_heads,
            "q_by_head": q_by_head, "k_by_head": k_by_head, "v_by_head": v_by_head,
            "q_by_head_prerope": q_by_head_prerope, "k_by_head_prerope": k_by_head_prerope,
            "head_output_by_head": head_output_by_head,
            "after_attn_residual": [downsample_1d(after_attn_residual[t], HIDDEN_STRIP) for t in range(seq_len)],
            "ln2_weight": downsample_1d(to_float(layer.post_attention_layernorm.weight), HIDDEN_STRIP),
            "ln2_out": [downsample_1d(ln2_outputs[li][t], HIDDEN_STRIP) for t in range(seq_len)],
            "moe_output": [downsample_1d(moe_out[t], HIDDEN_STRIP) for t in range(seq_len)],
            "layer_output": [downsample_1d(layer_out[t], HIDDEN_STRIP) for t in range(seq_len)],
        })

    # ---- self-validation on the first prompt (full precision, pre-downsample): only the correct
    # combine convention reconstructs the real MoE delta. DeepSeek uses RAW top-k weights
    # (norm_topk_prob=False) + additive gate-free shared experts. ----
    recon_errs = []  # best-convention rel_err per reconciled quantity; the smoke asserts on this
    if reconcile:
        li = moe_layer_indices[0]
        for t in [0, seq_len - 1]:
            h_t = gate_inputs[li][0][t]
            tt = ffn_layers_trace[li]["tokens"][t]
            routed_raw = sum(w * torch.nn.functional.linear(silu(torch.nn.functional.linear(h_t, weight_cache[(li, e)][0])) * torch.nn.functional.linear(h_t, weight_cache[(li, e)][1]), weight_cache[(li, e)][2])
                             for e, w in zip(tt["top_experts"], tt["top_weights"]))
            wsum = sum(tt["top_weights"])
            shared = mlp_forward(model.model.layers[li].mlp.shared_experts, h_t)
            real = hidden_states_all[li + 1][t] - (hidden_states_all[li][t] + attn_outputs[li][t])
            rel_raw = (routed_raw + shared - real).norm().item() / (real.norm().item() + 1e-9)
            rel_norm = (routed_raw / wsum + shared - real).norm().item() / (real.norm().item() + 1e-9)
            recon_errs.append(min(rel_raw, rel_norm))
            print(f"    [reconcile] MoE layer {li} tok {t}: RAW+shared rel_err={rel_raw:.4f} | NORM+shared rel_err={rel_norm:.4f} "
                  f"-> DeepSeek (norm_topk_prob=False) should favour RAW")
        # dense layer reconciliation
        for li_d in dense_layer_indices:
            t = seq_len - 1
            recon_dense = mlp_forward(model.model.layers[li_d].mlp, ln2_outputs[li_d][t])
            real = hidden_states_all[li_d + 1][t] - (hidden_states_all[li_d][t] + attn_outputs[li_d][t])
            rel = (recon_dense - real).norm().item() / (real.norm().item() + 1e-9)
            recon_errs.append(rel)
            print(f"    [reconcile] dense layer {li_d} token {t}: FFN recompute vs real delta  rel_err={rel:.4f} (expect <~0.02)")

    trace = {
        "prompt": prompt,
        "model_id": MODEL_ID,
        "num_layers": num_layers,
        "num_experts": num_experts,
        "top_k_experts": top_k_experts,
        "hidden_size": hidden_size,
        "intermediate_size": moe_intermediate_size,
        "shared_experts": n_shared_experts,
        "dense_layer_indices": dense_layer_indices,
        "tokens": [{"index": i, "text": s} for i, s in enumerate(token_strs)],
        "layers": ffn_layers_trace,
        "next_token_candidates": next_token_candidates,
        "router_matrices": router_matrices,
        "hidden_vectors": hidden_vectors,
        "expert_weights": expert_weights,
        "expert_outputs": expert_outputs,
        "shared_expert_weights": shared_expert_weights,
        "shared_expert_outputs": shared_expert_outputs,
        "dense_ffn": dense_ffn,
        "grid_dims": {"router": list(ROUTER_GRID), "hidden_strip": HIDDEN_STRIP, "expert": list(EXPERT_GRID)},
        "layer_flow": {
            "num_attention_heads": num_heads_attn,
            "head_dim": head_dim,
            "has_qk_norm": False,
            "embed_strip": embed_strip,
            "per_layer": per_layer_flow,
            "grid_dims": {"attn": list(ATTN_GRID)},
        },
    }
    if reconcile:
        trace["_reconcile"] = recon_errs  # smoke-only; the sweep calls reconcile=False so it never ships
    return trace


In [ ]:
# Can't execute this notebook locally to verify the hook wiring -- run one prompt first and
# sanity-check shapes/values (and the reconciliation diagnostic) before the full 12-prompt sweep
# on a 16B model. The reconcile=True pass self-validates the untestable deep extraction: only the
# correct combine convention (raw top-k weights + additive shared experts) reproduces the real
# MoE delta, so a small rel_err confirms the whole expert/shared/dense recompute path.
smoke_trace = extract_for_prompt(PROMPTS["math"][0], reconcile=True)
print(f"tokens: {[t['text'] for t in smoke_trace['tokens']]}")

layers = smoke_trace["layers"]  # rich schema uses OLMoE-style top-level layers (+ is_dense flag)
dense = [l["layer"] for l in layers if l["is_dense"]]
moe = [l["layer"] for l in layers if not l["is_dense"]]
print(f"layers: {len(layers)} (expect {num_layers}); dense: {dense} (expect [0]); moe: {len(moe)} (expect {num_layers - 1})")
print(f"layer 1 tok 0 top_experts: {layers[1]['tokens'][0]['top_experts']} (expect {top_k_experts} ids)")
print(f"layer 1 tok 0 all_probs sums to: {sum(layers[1]['tokens'][0]['all_probs']):.4f} (expect ~1.0)")
print(f"layer 1 tok 0 top_weights sum: {sum(layers[1]['tokens'][0]['top_weights']):.4f} (raw, NOT renormalized -- matches OLMoE)")

# rich-schema shape checks (these are the fields the deep math modals consume)
lf = smoke_trace["layer_flow"]
pl1 = lf["per_layer"][1]
print(f"layer_flow: {lf['num_attention_heads']} heads x {lf['head_dim']} dim, has_qk_norm={lf['has_qk_norm']}")
print(f"per_layer[1] attn maps: {len(pl1['attn_probs_all_heads'])} heads x "
      f"{len(pl1['attn_probs_all_heads'][0])}x{len(pl1['attn_probs_all_heads'][0][0])} (expect {num_heads_attn} x seq x seq)")
print(f"router_matrices[0]={smoke_trace['router_matrices'][0]} (None -> dense layer 0); "
      f"router_matrices[1] shape {len(smoke_trace['router_matrices'][1])}x{len(smoke_trace['router_matrices'][1][0])}")
print(f"dense_ffn keys: {list(smoke_trace['dense_ffn'].keys())}; "
      f"dense intermediate_size={smoke_trace['dense_ffn']['0']['intermediate_size']} (expect 10944)")
print(f"shared_expert_weights layers: {len(smoke_trace['shared_expert_weights'])} "
      f"(expect {num_layers - 1}); shared intermediate_size={smoke_trace['shared_expert_weights']['1']['intermediate_size']}")
print(f"expert_weights active pairs: {len(smoke_trace['expert_weights'])}; "
      f"expert_outputs entries: {len(smoke_trace['expert_outputs'])}")
print(f"top next-token prediction: {smoke_trace['next_token_candidates'][0]}")

# causal-structure eyeball on a real attention map (layer 1, head 0): strictly-upper-triangular
# mass must be ~0. This is the one path the reconcile does NOT transitively cover for DeepSeek.
amap = pl1["attn_probs_all_heads"][0]
upper = sum(amap[i][j] for i in range(len(amap)) for j in range(i + 1, len(amap)))
print(f"layer 1 head 0 strictly-upper-tri attn mass: {upper:.4f} (expect ~0 for causal masking)")

assert dense == [0]
assert layers[0]["tokens"] is None
assert len(moe) == num_layers - 1
assert len(layers[1]["tokens"][0]["top_experts"]) == top_k_experts
assert len(pl1["attn_probs_all_heads"]) == num_heads_attn
assert smoke_trace["dense_ffn"]["0"]["intermediate_size"] == 10944
assert len(smoke_trace["shared_expert_weights"]) == num_layers - 1
assert upper < 0.05, "attention map is not causal -- outputs.attentions may be wrong; recompute from q/k"
# HALT before the expensive 12-prompt sweep if the deep extraction doesn't reconstruct the model:
assert max(smoke_trace["_reconcile"]) < 0.1, (
    f"RECONCILE FAILED {smoke_trace['_reconcile']} -- the expert/shared/dense recompute does not match "
    f"the real model output. Do NOT run the full sweep; fix the extraction first."
)
print(f"Smoke test passed. reconcile best-convention rel_errs: {[round(x, 4) for x in smoke_trace['_reconcile']]}")


In [ ]:
all_traces = []
for domain, prompts in PROMPTS.items():
    for prompt in prompts:
        print(f"[{len(all_traces) + 1}/{total_prompts}] ({domain}) extracting: {prompt!r}")
        trace = extract_for_prompt(prompt)
        trace["domain"] = domain
        all_traces.append(trace)
        print(f"    tokens={[t['text'] for t in trace['tokens']]}  top_pred={trace['next_token_candidates'][0]}")

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"prompts": all_traces}, f)

print(f"\nWrote {len(all_traces)} prompts to {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e6:.2f} MB)")

## UMAP: per-domain routed-expert activation

Reuses the top-k routing decisions already captured in `all_traces` above (no extra forward
passes), same method as `extract_routing_trace.ipynb` cell 8. Routed experts only -- the 2
shared experts fire unconditionally on every token regardless of domain, so they carry no
discriminative signal for a per-domain activation-rate embedding. Layer 0 (dense, no
routing) is skipped when accumulating counts.

In [ ]:
import umap

NUM_LAYERS = num_layers
NUM_EXPERTS = num_experts
domain_to_idx = {d: i for i, d in enumerate(domains)}

# activation_counts[layer][expert][domain_idx] = count of this domain's tokens with this
# expert in top-k at this layer. tokens_per_domain[domain_idx] = total tokens seen for that
# domain (same at every layer, since every layer sees the same tokenized prompts). Layer 0
# is dense (tokens=None, no routing) and simply never gets any counts added to it.
activation_counts = np.zeros((NUM_LAYERS, NUM_EXPERTS, len(domains)), dtype=np.float64)
tokens_per_domain = np.zeros(len(domains), dtype=np.float64)

expert_token_scores = defaultdict(list)  # (layer, expert) -> [(weight, token, domain, prompt)]

for trace in all_traces:
    d_idx = domain_to_idx[trace["domain"]]
    n_tokens = len(trace["tokens"])
    tokens_per_domain[d_idx] += n_tokens
    for layer_trace in trace["layers"]:
        if layer_trace["tokens"] is None:  # dense layer, no routing
            continue
        li = layer_trace["layer"]
        for tt in layer_trace["tokens"]:
            token_text = trace["tokens"][tt["token_index"]]["text"]
            for e, w in zip(tt["top_experts"], tt["top_weights"]):
                activation_counts[li, e, d_idx] += 1
                expert_token_scores[(li, e)].append((w, token_text, trace["domain"], trace["prompt"]))

tokens_per_domain[tokens_per_domain == 0] = 1  # guard divide-by-zero
activation_rates = activation_counts / tokens_per_domain[None, None, :]
expert_vectors = activation_rates.reshape(NUM_LAYERS * NUM_EXPERTS, len(domains))

point_layer_ids = np.repeat(np.arange(NUM_LAYERS), NUM_EXPERTS)
point_expert_ids = np.tile(np.arange(NUM_EXPERTS), NUM_LAYERS)

# Never-activated (layer, expert) pairs are all-zero and undefined under the cosine metric
# (0/0 -> NaN) -- exclude from the projection, report separately as excluded_experts. This
# also naturally excludes all of layer 0's (dense, no experts) NUM_EXPERTS placeholder rows.
active_mask = expert_vectors.sum(axis=1) > 0
active_vectors = expert_vectors[active_mask]

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, metric="cosine", n_jobs=1)
active_embedding = reducer.fit_transform(active_vectors)

assert not np.isnan(active_embedding).any(), (
    "UMAP produced NaN coordinates even after excluding all-zero rows -- inspect "
    "active_vectors for degenerate rows, or re-run with metric='euclidean'."
)

print(f"Built {expert_vectors.shape[0]} (layer, expert) vectors across {len(domains)} domains.")
print(f"UMAP embedding shape: {active_embedding.shape} ({int(active_mask.sum())} active of {expert_vectors.shape[0]} total pairs)")

TOP_K_TOKENS = 8
active_indices = np.flatnonzero(active_mask)

umap_points = []
for row, i in enumerate(active_indices):
    layer_id = int(point_layer_ids[i])
    expert_id = int(point_expert_ids[i])
    vec = expert_vectors[i]
    dominant_domain = domains[int(np.argmax(vec))]

    samples = sorted(expert_token_scores.get((layer_id, expert_id), []), key=lambda item: -item[0])[:TOP_K_TOKENS]
    top_tokens = [
        {"token": tok, "score": round(float(score), 4), "domain": dom, "prompt": prompt}
        for score, tok, dom, prompt in samples
    ]

    umap_points.append({
        "layer_id": layer_id,
        "expert_id": expert_id,
        "x": round(float(active_embedding[row, 0]), 4),
        "y": round(float(active_embedding[row, 1]), 4),
        "dominant_domain": dominant_domain,
        "domain_activation_rate": {d: round(float(vec[j]), 4) for j, d in enumerate(domains)},
        "top_tokens": top_tokens,
    })

excluded_experts = [
    {"layer_id": int(point_layer_ids[i]), "expert_id": int(point_expert_ids[i])}
    for i in np.flatnonzero(~active_mask)
]

assert len(umap_points) + len(excluded_experts) == NUM_LAYERS * NUM_EXPERTS

umap_data = {
    "domains": domains,
    "num_layers": NUM_LAYERS,
    "num_experts": NUM_EXPERTS,
    "dense_layers": dense_layer_indices,
    "points": umap_points,
    "excluded_experts": excluded_experts,
}

with open(UMAP_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(umap_data, f, ensure_ascii=False, allow_nan=False, indent=2)

print(f"Wrote {UMAP_OUT_PATH} ({len(umap_points)} points, {len(excluded_experts)} excluded pairs)")

In [ ]:
try:
    from google.colab import files
    files.download(OUT_PATH)
    files.download(UMAP_OUT_PATH)
except ImportError:
    print("Not running in Google Colab -- skipping auto-download.")
    print(f"Files were written locally at: {OUT_PATH} and {UMAP_OUT_PATH}")